# ランダムな 4x4 エルミート行列の量子位相推定

著者: Takumi Kato (Blueqat inc.), Maho Nakata (Riken), Shinya Morino, Seiya Sugo (Quemix inc.), Yuichiro Minato (Blueqat inc.)

[前回](114_pea2_en.ipynb)は、2x2 のエルミート行列の固有値を計算しました。今回は、ランダムに生成した 4x4 のエルミート行列に対する量子位相推定を行います。

エルミート行列の固有値を計算することは、量子力学における物理量を求める上で有用であり、量子化学や量子シミュレーションといった分野で幅広く応用されています。

## 実装
まず、必要なライブラリをインポートします。今回はデフォルトの(tensornet)バックエンドを使用します。

In [ ]:
import math
import cmath
import random
import numpy as np

from blueqat import *
from blueqat.utils import X, Y, Z, I

pi = math.pi

# Version check for Blueqat
try:
    Circuit().r(0.1)[0].run()
except AttributeError:
    raise ImportError('Blueqat version is old.')

In [ ]:
# Qgate backend has been removed in the new blueqat SDK; the default
# tensornet backend is used instead, so no separate installation check is needed.

In [ ]:
# If ImportError is occured, you didn't install Qgate.
# You can use Qgate without CUDA, but it's recommended on Linux environment.
# https://github.com/shinmorino/qgate#build--install

# To install pre-built package, run following command.

# For Python 3.8 user:
# !pip install https://github.com/shinmorino/qgate/raw/gh-pages/packages/0.2/qgate-0.2.2-cp38-cp38-manylinux1_x86_64.whl

# For Python 3.7 user:
# !pip install https://github.com/shinmorino/qgate/raw/gh-pages/packages/0.2/qgate-0.2.2-cp37-cp37m-manylinux1_x86_64.whl

# For Python 3.6 user:
# !pip install https://github.com/shinmorino/qgate/raw/gh-pages/packages/0.2/qgate-0.2.2-cp36-cp36m-manylinux1_x86_64.whl

# Don't install and use numba backend
# BlueqatGlobalSetting.set_default_backend('numba')

次に、ランダムなエルミート行列 $\hat H$ を作成します。

量子位相推定によって固有値を得るためには、以下も必要です。
- 求めたい固有値に対応する $\hat H$ の固有ベクトルを与える量子回路
- Controlled-$e^{i2\pi \hat H 2^n}$ を与える量子回路

これらを作成していきます。

エルミート行列は $\hat H = P D P^\dagger$ のように分解できます。ここで $P$ はユニタリ行列、$D$ は実対角行列です。

この場合、$D$ の対角成分は $\hat H$ の固有値であり、$P$ の各列は $\hat H$ の固有ベクトルです。今回は 4x4 のエルミート行列を考えているため、ランダムに生成した SU(4) 回路が必要になります。このような SU(4) 回路は、4つの SU(2) 行列と、3つのパラメータを持つ行列から生成できます。([arXiv:quant-ph/0507171](https://arxiv.org/abs/quant-ph/0507171) を参照)

![circuit](../img/115_1.png)

ここで $e^{iH} = e^{i(k_1 \sigma_{XX} + k_2 \sigma_{YY} + k_3 \sigma_{ZZ})}$ であり、
$k_i$ はパラメータ、$\sigma_{**}$ はパウリ行列のクロネッカー積です。

それでは、以下を返す関数を定義しましょう。

- エルミート行列 $\hat H$
- 求めたい固有値
- $\hat H$ の固有ベクトルを与える回路

In [ ]:
def is_hermitian(mat):
    """Check whether mat is Hermitian"""
    return np.allclose(mat, mat.T.conjugate())

def circuit_to_unitary(c):
    """Get a unitary matrix from circuit"""
    n_qubits = c.n_qubits
    def bits(n):
        return tuple(i for i in range(8) if (1 << i) & n)
    vecs = []
    for i in range(2**n_qubits):
        c0 = Circuit().x[bits(i)]
        c0 += c
        vecs.append(c0.run().reshape((-1, 1)))
    return np.hstack(vecs)

def rand_2pi():
    return random.random() * 2 * pi

def rand_eigval():
    return random.random() * 2 - 1

def random_su4_circuit():
    c = Circuit()
    c.u(rand_2pi(), rand_2pi(), rand_2pi())[0]
    c.u(rand_2pi(), rand_2pi(), rand_2pi())[1]
    c.cx[0, 1].rx(rand_2pi())[0].h[0].rz(rand_2pi())[1]
    c.cx[0, 1].s[0].h[0].rz(rand_2pi())[1]
    c.cx[0, 1].sdg[0].h[0].sdg[0].s[1].h[1].s[1]
    c.u(rand_2pi(), rand_2pi(), rand_2pi())[0]
    c.u(rand_2pi(), rand_2pi(), rand_2pi())[1]
    return c

def random_hermitian():
    """Make random Hermitian and returns triplet
    (Hermitian, eigenvalues, Circuit for making eigenvectors).
    """
    # Generate random eigenvalue
    eigvals = [rand_eigval(), rand_eigval(), rand_eigval(), rand_eigval()]
    eigvals.sort()
    su4 = random_su4_circuit()
    # Make Hermitian from them
    p = circuit_to_unitary(su4)
    hermitian = p @ np.diag(eigvals) @ p.T.conjugate()
    # Check it is Hermitian
    assert is_hermitian(hermitian)
    # returns Hermitian, eigenvalue, circuit
    return hermitian, eigvals, su4

では、エルミート行列を作ってみましょう。

In [ ]:
H, eigvals, su4 = random_hermitian()
print('H:')
print(H)
print()
print('Eigenvalues:')
print(eigvals)
print()
print('Eigenvectors (P = [v1 v2 v3 v4]):')
P = circuit_to_unitary(su4)
print(P)
print()
print('P D P† = H? (D = diagonal matrix of eigenvalues):', np.allclose(P @ np.diag(eigvals) @ P.T.conjugate(), H))

`theta, phi, lam` と U ゲートを使って固有ベクトルを作ることができます。

In [ ]:
vec = su4.run()
print(vec)

`H vec = E vec` を確認します。これは、`E, vec` が H の固有値と固有ベクトルの組であることを意味します。

In [ ]:
np.allclose(np.dot(H, vec), eigvals[0] * vec)

準備が整いました。量子位相推定を実装していきます。量子回路を作り、量子位相推定によって `E` を計算します。

In [ ]:
def iqft(c, q0, n_qubits):
    """Add inversed quantum Fourier transform operations to q0-th - (q0 + n_qubits)-th qubits of the circuit `c`"""
    for i in reversed(range(n_qubits)):
        angle = -0.5
        for j in range(i + 1, n_qubits):
            c.cr(angle * pi)[q0 + j, q0 + i]
            angle *= 0.5
        c.h[q0 + i]
    return c


def apply_cu(c, ctrl, su4, eigvals, n):
    """Append Controlled-U^(2^n) to the circuit `c`.
    Controll qubit is `ctrl`, target qubit is 0 and 1.
    
    This function requires eigenvalue `eigval` as an argument.
    We make Controlled-U^(2^n) by using eigenvalue. You may feel we're cheating.
    You can make approximate Controlled-U^(2^n) circuit without eigenvalue,
    for example by using Suzuki-Trotter decomposition. In this case, you have to consider about precision.
    Generally, making efficient and high-precision Controlled-U^(2^n) circuit without cheating is difficult.
    """
    bias = 0.25 * sum(eigvals)
    p, q, r, s = [v - bias for v in eigvals]
    p1 = pi * (r + s)
    p2 = pi * (q + s)
    p3 = pi * (q + r)
    c += su4.dagger()
    c.crz(p1 * (2**n))[ctrl, 1]
    c.crz(p2 * (2**n))[ctrl, 0]
    c.ccx[ctrl, 1, 0].crz(p3 * (2**n))[ctrl, 0].ccx[ctrl, 1, 0]
    c.rz(pi * bias * (2**n))[ctrl]
    c += su4
    return c

def qpe_circuit(initial_circuit, eigvals, su4, precision):
    """Returns quantum phase estimation circuit"""
    c = initial_circuit + su4
    c.h[2:2 + precision]
    for i in range(precision):
        apply_cu(c, i + 2, su4, eigvals, i)
    iqft(c, 2, precision)
    return c

量子位相推定の量子回路を見てみましょう。

In [ ]:
%matplotlib inline
qpe_circuit(Circuit(), eigvals, su4, 4).run_with_draw()

次に、観測結果から固有値を計算する関数を作ります。

In [ ]:
def run_qpe(c, shots=1000, max_candidates=5):
    """Run the circuit for quantum phase estimation and returns candidates of eigenvalue.
    shots: The number of running quantum circuit, max_candidates: Maximum number of candidates
    """
    # Use the explicit "statevector" backend: the default "tensornet" backend can run out of
    # memory / crash for the larger circuits built at high precision below.
    cnt = c.m[2:].run(shots=shots, backend="statevector")

    # Convert measured result to floating point value
    def to_value(k):
        # The new SDK's shot-result bitstrings are ordered with qubit 0 as the rightmost
        # character (opposite of the old convention), so reverse before parsing.
        k = k[::-1]
        k = k[2:] # Drop unnecessary element
        val = 0 # Value
        a = 1.0 
        for ch in k:
            if ch == '1':
                val += a
            a *= 0.5
        if val > 1:
            # When the phase > π, subtract 2π (phase is negative).
            val = val - 2
        return val

    return [(to_value(k), v) for k, v in cnt.most_common(max_candidates)]

これで量子位相推定を実行できます。結果を見てみましょう。

In [ ]:
print('Eigenvalue (expected):', eigvals[0]) # It's expected value. We desire a value closed to it.
# We'll run small to high precision for comparison.
for precision in range(3, 16):
    print(precision, 'bit precision:')
    c = qpe_circuit(Circuit(), eigvals, su4, precision)
    result = run_qpe(c, 1000, 3)
    for value, count in result:
        # Display a number of observation, obtained eigenvalue and a deviation from true eigenvalue.
        print(f'{count:<5}{value:<18}(deviation: {value - eigvals[0]: .3e})')
    print('')

得られた値は真の固有値に近い値になっています。

次に、2番目の固有値を計算します。

In [ ]:
print('Eigenvalue (expected):', eigvals[1]) # It's expected value. We desire a value closed to it.
# We'll run small to high precision for comparison.
for precision in range(3, 16):
    print(precision, 'bit precision:')
    c = qpe_circuit(Circuit().x[0], eigvals, su4, precision)
    result = run_qpe(c, 1000, 3)
    for value, count in result:
        # Display a number of observation, obtained eigenvalue and a deviation from true eigenvalue.
        print(f'{count:<5}{value:<18}(deviation: {value - eigvals[1]: .3e})')
    print('')

では、3番目の固有値を計算してみましょう。

In [ ]:
print('Eigenvalue (expected):', eigvals[2]) # It's expected value. We desire a value closed to it.
# We'll run small to high precision for comparison.
for precision in range(3, 16):
    print(precision, 'bit precision:')
    c = qpe_circuit(Circuit().x[1], eigvals, su4, precision)
    result = run_qpe(c, 1000, 3)
    for value, count in result:
        # Display a number of observation, obtained eigenvalue and a deviation from true eigenvalue.
        print(f'{count:<5}{value:<18}(deviation: {value - eigvals[2]: .3e})')
    print('')

最後の固有値は次の通りです。

In [ ]:
print('Eigenvalue (expected):', eigvals[3]) # It's expected value. We desire a value closed to it.
# We'll run small to high precision for comparison.
for precision in range(3, 16):
    print(precision, 'bit precision:')
    c = qpe_circuit(Circuit().x[0, 1], eigvals, su4, precision)
    result = run_qpe(c, 1000, 3)
    for value, count in result:
        # Display a number of observation, obtained eigenvalue and a deviation from true eigenvalue.
        print(f'{count:<5}{value:<18}(deviation: {value - eigvals[3]: .3e})')
    print('')